# Test Inference v3
Workflow multi-agentico con **Pydantic `with_structured_output()`** — zero regex, zero `json.loads` manuali.
Modello: `google/gemma-4-31b-it`

In [1]:
import os, sys, json
os.chdir("/workspaces/hackaton-UNIBG-repo")
sys.path.insert(0, "src")
from dotenv import load_dotenv; load_dotenv()
print("API Key:", "OK" if os.getenv("OPEN_ROUTER_KEY") else "MANCA")

API Key: OK


## Modelli Pydantic
Definiscono la struttura garantita degli output LLM.

In [2]:
from typing import Any, Dict, List
from pydantic import BaseModel, Field

class FieldDefinition(BaseModel):
    field: str = Field(description="Nome del campo")
    type: str = Field(description="Tipo: string, number, date, list, object")
    sensitive: bool = Field(description="True se contiene dati personali")
    constraints: str = Field(description="Vincoli del campo")

class DocumentExtraction(BaseModel):
    schema_def: List[FieldDefinition] = Field(description="Schema del documento")
    data: Dict[str, Any] = Field(description="Dati estratti dal documento")

class ValidatedRecords(BaseModel):
    records: List[Dict[str, Any]] = Field(description="Record validati e corretti")

print("Modelli Pydantic definiti")

Modelli Pydantic definiti


## 1. OCR del PDF

In [3]:
from inferencev3 import extract_text_from_pdf

raw_text = extract_text_from_pdf("dataset/invoice-001.pdf")
print(raw_text[:1200])

2026-05-28 21:01:24.441 | INFO     | inferencev3:extract_text_from_pdf:49 - Estrazione OCR da: dataset/invoice-001.pdf


--- Page 1 ---
Invoice no: 51109338 et Om,

Date of issue: 04/13/2013 Ques

TECHSHOPY

Seller: Client:

Andrews, Kirby and Valdez Becker Ltd

58861 Gonzalez Prairie 8012 Stewart Summit Apt. 455
Lake Daniellefurt, IN 57228 North Douglas, AZ 95355

Tax Id: 945-82-2137 Tax Id: 942-80-0517

IBAN: GB75MCRL06841367619257

ITEMS
No. Description Qty UM Net price Net worth VAT [%] Gross
worth
1, CLEARANCE! Fast Dell Desktop 3,00 each 209,00 627,00 10% 689,70
Computer PC DUAL CORE
WINDOWS 10 4/8/16GB RAM
2. HP T520 Thin Client Computer 5,00 each 37,75 188,75 10% 207,63
AMD GX-212JC 1.2GHz 4GB RAM
TESTED !!READ BELOW!!
3: gaming pc desktop computer 1,00 each 400,00 400,00 10% 440,00
4. 12-Core Gaming Computer 3,00 each 464,89 1 394,67 10% 1 534,14
Desktop PC Tower Affordable
GAMING PC 8GB AMD Vega RGB
5. Custom Build Dell Optiplex 9020 5,00 each 221,99 1 109,95 10% 1 220,95
MT i5-4570 3.20GHz Desktop
Computer PC
6. Dell Optiplex 990 MT Computer 4,00 each 269,95 1 079,80 10% 1 187,78
PC Quad Core 

## 2. Agente Analizzatore — `with_structured_output(DocumentExtraction)`

In [5]:
from inferencev3 import build_llm

llm = build_llm(0.2)
structured_llm = llm.with_structured_output(DocumentExtraction)

prompt = (
    "Analizza il seguente testo OCR estratto da un PDF.\n\n"
    "TESTO OCR:\n" + raw_text + "\n\n"
    "Il tuo compito:\n"
    "1. Deduci la struttura del documento (NON fare assunzioni sul tipo).\n"
    "2. Per ogni campo, indica nome, tipo, se e' sensibile, e eventuali vincoli.\n"
    "3. Estrai i dati dal documento nel formato dedotto.\n\n"
    "Marca come sensitive: true TUTTI i campi con dati personali."
)

extraction: DocumentExtraction = structured_llm.invoke(prompt)

schema_list = [fd.model_dump() for fd in extraction.schema_def]
print("=== SCHEMA ===")
print(json.dumps(schema_list, indent=2))
print("\n=== DATI SORGENTE ===")
print(json.dumps(extraction.data, indent=2))

KeyboardInterrupt: 

## 3. Agente Generatore (5 record sintetici)
Il generatore usa `create_agent()` perché produce array JSON a struttura variabile.

In [ ]:
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage

N = 1
schema_json = json.dumps(schema_list, indent=2)
source_json = json.dumps(extraction.data, indent=2)

llm2 = build_llm(0.7)
generator = create_agent(
    model=llm2,
    tools=[],
    system_prompt=(
        "Sei un generatore di dati sintetici. Genera ESATTAMENTE " + str(N) + " record. "
        "Rispetta lo schema, sostituisci campi sensitive con dati fittizi, "
        "mantieni coerenza interna. Output SOLO array JSON."
    ),
)

prompt = (
    "SCHEMA:\n" + schema_json +
    "\n\nDATI:\n" + source_json +
    "\n\nGenera " + str(N) + " record."
)

result = generator.invoke({"messages": [HumanMessage(content=prompt)]})
generated_raw = result["messages"][-1].content
print("Generatore output (primi 500 char):", generated_raw[:500])

## 4. Agente Validatore — `with_structured_output(ValidatedRecords)`

In [ ]:
llm3 = build_llm(0.1)
structured_val = llm3.with_structured_output(ValidatedRecords)

prompt = (
    "Valida i seguenti record sintetici contro lo schema fornito.\n\n"
    "SCHEMA e constraints:\n" + schema_json +
    "\n\nRECORD DA VALIDARE:\n" + generated_raw +
    "\n\nPer OGNI record: verifica campi, tipi, constraints. "
    "Correggi errori di calcolo. Rimuovi record invalidi."
)

validation: ValidatedRecords = structured_val.invoke(prompt)
validated = validation.records

print(f"Validati: {len(validated)} record")
print(json.dumps(validated[0], indent=2)[:600])

## 5. Confronto sorgente vs sintetico
I dati ora sono oggetti Python nativi — nessun parsing richiesto.

In [ ]:
from IPython.display import display, HTML

html = "<table><tr><th>Campo</th><th>Sorgente</th><th>Sintetico</th></tr>"
for fd in extraction.schema_def:
    name = fd.field
    orig = str(extraction.data.get(name, ""))[:40]
    synth = str(validated[0].get(name, ""))[:40] if validated else ""
    color = "red" if fd.sensitive else "green"
    html += f"<tr><td style='color:{color}'>{name}</td><td>{orig}</td><td>{synth}</td></tr>"
html += "</table>"
display(HTML(html))

## Bonus: workflow completo in un colpo solo

In [ ]:
from inferencev3 import build_graph

app = build_graph()

result = app.invoke({
    "messages": [
        HumanMessage(content="5"),
        HumanMessage(content="run_id:notebook_v3"),
    ],
    "raw_text": raw_text,
    "document_schema": "",
    "source_data": "",
    "generated_data": "",
    "validated_data": "",
    "final_data": [],
    "errors": [],
})

print(f"Completato: {len(result['final_data'])} record salvati in output/notebook_v3/")